Quantum State Tomography

In [ ]:
# QST measurement simulation function
import itertools as it
import numpy as np
from qutip import *
import pandas as pd
from scipy.linalg import hadamard

class QSTSim:

    def __init__(self, nqubits, shots, rho):
        self.nqubits = nqubits
        self.shots = shots
        self.hadamard = hadamard(2**self.nqubits)
        self.rho = rho

    def calc_probs(self):
        settings = list(it.product(["X", "Y", "Z"], repeat=self.nqubits))
        # returns all combnations for e.g. in 2 qubit case XX, XY, YZ so on
        
        # Z basis projectors
        Pz_plus = ket2dm(basis(2,0))  
        Pz_minus = ket2dm(basis(2,1))

        xplus = (basis(2,0) + basis(2,1)).unit()
        xminus = (basis(2,0) - basis(2,1)).unit()

        # X basis projectors
        Px_plus = ket2dm(xplus) # ket to density matrix
        Px_minus = ket2dm(xminus)

        # Y basis projectors
        p_i = (basis(2,0) + 1j*basis(2,1)).unit()
        m_i = (basis(2,0) - 1j*basis(2,1)).unit()

        Py_plus = ket2dm(p_i)
        Py_minus = ket2dm(m_i)

        proj_lookup = {"X": [Px_plus, Px_minus],
                "Y": [Py_plus, Py_minus],
                "Z": [Pz_plus, Pz_minus]}

        outcomes = list(it.product([0,1], repeat=self.nqubits)) # one time gen of e.g. [00, 01, 10, 11] for 2 qubits. This represents +1, -1 measurements

        prob_dict = {}
        tens_mat = {}
        for setting in settings:
            projectors = []
            for outcome in outcomes:
                operators = []
                for qubit, b in enumerate(setting):
                    operators.append(proj_lookup[b][outcome[qubit]])
                projectors.append(tensor(*operators))                

            # compute probabilities
            probs = []

            for P in projectors:
                prob = (P * self.rho).tr() # tr(P*rho) = prob coress to P
                probs.append(prob)

            # ---- store results ----
            prob_dict[setting] = probs
            tens_mat[setting] = projectors   
        return prob_dict, tens_mat 

   
    def print_probs(self, prob_dict):
        rows = []
        for setting, probs in prob_dict.items():
            rows.append({
                "Basis": "".join(setting),
                "(+1,+1)": probs[0],
                "(+1,-1)": probs[1],
                "(-1,+1)": probs[2],
                "(-1,-1)": probs[3]
            })

        df = pd.DataFrame(rows)
        print("Measurement for the Rho state:", self.rho)
        print("Printing probabilities:\n",df.to_string(index=False, float_format="%.2f"))


    def measure_qubits(self):
        prob_dict, tensor_mat = self.calc_probs()
        row = list(it.product([-1,1], repeat=self.nqubits))
        idx = np.arange(len(row))

        expectationval_list = []
        count_list = []
        self.shots = int(self.shots/(3**self.nqubits)) # shots per basis setting, e.g. if 2 qubit ie 1000/3^2 = 1000/9 = 111 shots per basis setting
        for i, prob in enumerate(prob_dict.values()): # i is index of basis setting, prob is list of probabilities for that setting e.g [0.25, .25, .25, .25] for 2 qubits
            measure = list(np.random.choice(len(row), self.shots, p=prob)) # choice takes in the probabilities for that basis setting and generates a list of measurement outcomes (e.g. 1000 samples of 0,1,2,3 for 2 qubits) based on those probabilities
            counts = np.array([measure.count(x)/self.shots for x in idx]) # counts the number of times each outcome (0,1,2,3) occurs in the measure list and divides by total shots to get probabilities for each outcome based on the simulated measurements
            count_list.append(counts)
            expectationvals = np.dot(self.hadamard, counts) # frequency * hadamard to get expectation values for that basis setting
            expectationval_list.append(expectationvals)

        return expectationval_list, count_list   


    def print_counts(self, count_list):

        labels = ["(+1,+1)", "(+1,-1)", "(-1,+1)", "(-1,-1)"]

        rows = []
        for i, counts in enumerate(count_list):
            rows.append({
                "Basis #": i,
                "(+1,+1)": counts[0],
                "(+1,-1)": counts[1],
                "(-1,+1)": counts[2],
                "(-1,-1)": counts[3]
            })

        df = pd.DataFrame(rows)
        print("\nMeasured probabilities (counts):")
        print(df.to_string(index=False, float_format="%.3f"))


    def print_expectations(self, expect_list):

        settings = list(it.product(["X","Y","Z"], repeat=self.nqubits))
        rows = []
        for i,(setting,e) in enumerate(zip(settings, expect_list)):
            labels = []
            for comb in it.product(["I","M"], repeat=self.nqubits): # M is just placeholder to replace the right operator among X, Y, Z
                label = ""
                for j,c in enumerate(comb):
                    label += "I" if c=="I" else setting[j]
                labels.append(label)

            row = {"Basis": "".join(setting)}

            for j,l in enumerate(labels):
                row[l] = e[j]

            rows.append(row)

        df = pd.DataFrame(rows)
        print("\nExpectation values:")
        print(df.to_string(index=False, float_format="%.3f"))  

def test_calc_probs():
    sim = QSTSim(2, 1000, tensor(basis(2,0), basis(2,0)) * tensor(basis(2,0), basis(2,0)).dag()) #|00><00|
    prob_dict, tensor_mat = sim.calc_probs()
    sim.print_probs(prob_dict)


def test_measure():
    sim = QSTSim(2, 1000, tensor(basis(2,0), basis(2,0)) * tensor(basis(2,0), basis(2,0)).dag()) #|00><00|
    expect_list, count_list = sim.measure_qubits()
    sim.print_counts(count_list)
    sim.print_expectations(expect_list) # expectations = Hadamard × probabilities
    print("Dim of counts", np.array(count_list).shape)
    print("Dim of expect_list", np.array(expect_list).shape)

test_calc_probs()
test_measure()

Measurement for the Rho state: Quantum object: dims = [[2, 2], [2, 2]], shape = (4, 4), type = oper, isherm = True
Qobj data =
[[1. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
Printing probabilities:
 Basis  (+1,+1)  (+1,-1)  (-1,+1)  (-1,-1)
   XX     0.25     0.25     0.25     0.25
   XY     0.25     0.25     0.25     0.25
   XZ     0.50     0.00     0.50     0.00
   YX     0.25     0.25     0.25     0.25
   YY     0.25     0.25     0.25     0.25
   YZ     0.50     0.00     0.50     0.00
   ZX     0.50     0.50     0.00     0.00
   ZY     0.50     0.50     0.00     0.00
   ZZ     1.00     0.00     0.00     0.00

Measured probabilities (counts):
 Basis #  (+1,+1)  (+1,-1)  (-1,+1)  (-1,-1)
       0    0.243    0.234    0.252    0.270
       1    0.288    0.189    0.243    0.279
       2    0.459    0.000    0.541    0.000
       3    0.261    0.279    0.243    0.216
       4    0.225    0.315    0.252    0.207
       5    0.523    0.000    0.477    0.000
       6    0.514 